# Blue Sky API Report
Hannah Jamison

October 21, 2026

**Research Question:** Do accounts with more followers get more reposts?

**Hypothesis:** I would believe that if an account has a larger following, it would also have more reposts. 

This is because the account will reach more people in comparison to an account with less followers, therefore leading to more oppurtunities for their posts to be reposted.
In order to test my hypothesis I will need to gather the follower and repost variables of multiple accounts, varying in amounts of followers. Then compare those variables against eachother to confirm or deny my hypothesis. The data is found reliable due to it being taken directly from the Blue Sky data base. It could be unreliable due to the question only being tested with a small handful of accounts, it is not accounting for all accounts. So the results with all Blue Sky accounts could differ from the results of this report.

In [2]:
import requests
import pandas as pd

url = "https://public.api.bsky.app/xrpc/app.bsky.actor.getProfile"

params={
    "actor":"bsky.app"
}
response = requests.get(url, params=params, timeout=10)

print(response.status_code)

200


In [50]:
BASE = "https://public.api.bsky.app/xrpc"

accounts = [
    "aoc.bsky.social",
    "hankgreen.bsky.social",
    "weratedogs.com",
    "playstation.com"
]

profiles = []

for account in accounts:
    response = requests.get(
        BASE + "/app.bsky.actor.getProfile",
        params={"actor": account},
        timeout=15
    )

    print(account, response.status_code)
    response.raise_for_status()

    data = response.json()

    profiles.append({
        "did": data["did"],
        "handle": data["handle"],
        "followers": data.get("followersCount", 0)
    })

profiles_df = pd.DataFrame(profiles)
display(profiles_df)

aoc.bsky.social 200
hankgreen.bsky.social 200
weratedogs.com 200
playstation.com 200


,did,handle,followers
0,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,2210002
1,did:plc:kqbyr4gqt6p2l57htlsa4nha,hankgreen.bsky.social,847558
2,did:plc:rlr4e4zioxivpndzr4mqw4r2,weratedogs.com,562871
3,did:plc:3nfshkzomgboapasu6amkhui,playstation.com,173757


In [46]:
feed_url = BASE + "/app.bsky.feed.getAuthorFeed"
response = requests.get(
    feed_url,
    params={"actor":profiles_df.loc[0, "did"],
            "limit":30, "filer":"posts_no_replies"},
    timeout=15
)
response.raise_for_status()
sample = response.json()
print(sample.keys())
print(sample["feed"][0].keys())
print(sample["feed"][0]["post"].keys())

dict_keys(['feed', 'cursor'])
dict_keys(['post', 'reason'])
dict_keys(['uri', 'cid', 'author', 'record', 'embed', 'bookmarkCount', 'replyCount', 'repostCount', 'likeCount', 'quoteCount', 'indexedAt', 'labels'])


In [47]:
post_frames = []
feed_url = BASE + "/app.bsky.feed.getAuthorFeed"

for profile in profiles:
    print("Fetching posts for:", profile["handle"])

    response = requests.get(
        feed_url,
        params={
            "actor": profile["did"],
            "limit": 30,
            "filter": "posts_no_replies"
        },
        timeout=15
    )
    response.raise_for_status()

    feed = response.json().get("feed", [])

    rows = []

    for item in feed:
        post = item["post"]

        if post["author"]["did"] == profile["did"]:
            rows.append({
                "did": profile["did"],
                "handle": profile["handle"],
                "uri": post["uri"],
                "text": post["record"].get("text", ""),
                "reposts": post.get("repostCount", 0),
                "created_at": post["record"].get("createdAt")
            })

    print("Posts collected:", len(rows))
    post_frames.append(pd.DataFrame(rows))

posts_df = pd.concat(post_frames, ignore_index=True)

print("\nPosts by account:")
print(posts_df["handle"].value_counts())

display(posts_df.head())

Fetching posts for: aoc.bsky.social
Posts collected: 15
Fetching posts for: hankgreen.bsky.social
Posts collected: 22
Fetching posts for: weratedogs.com
Posts collected: 27
Fetching posts for: playstation.com
Posts collected: 30

Posts by account:
handle
playstation.com          30
weratedogs.com           27
hankgreen.bsky.social    22
aoc.bsky.social          15
Name: count, dtype: int64


,did,handle,uri,text,reposts,created_at
0,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Come with me to help our local Daisy Scouts ge...,736,2026-05-06T21:10:36.371Z
1,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,You ever try to code switch at work and fail? 😂,3939,2026-04-22T02:50:58.146Z
2,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Reading people. Body language. Knowing how to ...,1620,2026-02-22T17:07:34.084Z
3,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Let’s do a pre-snowfall AMA! ❄️\n\nHave any (r...,409,2026-02-22T16:37:51.221Z
4,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Also now that they have drafts here we are so ...,318,2026-02-21T17:24:04.185Z


I used ChatGPT to help figure out the correct code because orginally the results of that code was only the data from Playstation.

In [48]:
combined_df = pd.merge(
    posts_df,
    profiles_df[["did", "followers"]],
    on = "did", how="left", validate="many_to_one"
)
print(combined_df["handle"].value_counts(dropna=False))
display(combined_df.head(10))

handle
playstation.com          30
weratedogs.com           27
hankgreen.bsky.social    22
aoc.bsky.social          15
Name: count, dtype: int64


,did,handle,uri,text,reposts,created_at,followers
0,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Come with me to help our local Daisy Scouts ge...,736,2026-05-06T21:10:36.371Z,2210002
1,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,You ever try to code switch at work and fail? 😂,3939,2026-04-22T02:50:58.146Z,2210002
2,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Reading people. Body language. Knowing how to ...,1620,2026-02-22T17:07:34.084Z,2210002
3,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Let’s do a pre-snowfall AMA! ❄️\n\nHave any (r...,409,2026-02-22T16:37:51.221Z,2210002
4,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Also now that they have drafts here we are so ...,318,2026-02-21T17:24:04.185Z,2210002
5,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,"Apologies for the very Catholic post, but is a...",245,2026-02-21T17:23:36.834Z,2210002
6,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,Ocasio gang rise up 🙌🏽🇵🇷,3760,2026-02-09T01:49:50.383Z,2210002
7,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,I hope Dems begin to see that moderate or prog...,3030,2026-02-06T14:52:46.533Z,2210002
8,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,"It’s not about drugs. If it was, Trump wouldn’...",8335,2026-01-03T15:34:43.013Z,2210002
9,did:plc:p7gxyfr5vii5ntpwo7f6dhe2,aoc.bsky.social,at://did:plc:p7gxyfr5vii5ntpwo7f6dhe2/app.bsky...,You just know that “Did Women Ruin the Workpla...,3573,2025-11-13T15:26:20.892Z,2210002


In [49]:
summary_df = combined_df.groupby("handle").agg(
    followers=("followers", "first"),
    total_reposts=("reposts", "sum"),
    posts_collected=("uri", "nunique")
).reset_index()

display(summary_df)

,handle,followers,total_reposts,posts_collected
0,aoc.bsky.social,2210002,63933,15
1,hankgreen.bsky.social,847558,691,22
2,playstation.com,173757,506,30
3,weratedogs.com,562871,29785,27


I used ChatGPT to help understand how to use the .groupby function.

This data does show some agreement with my hypothesis, though it does not fully prove it. AOC does have the most followers and the most reposts, but when you look at the other accounts, it's a bit different. Even though weratedogs has less followers than Hank Green, they have 29,000 more reposts. The way this data is put together is through only a small handful of posts from each account, which does put a limitation on this data report. If you were to include every single post, from all of the accounts the number could end up different. Moving forward, I would probably make sure to pick accounts of a similar style (all news platforms, politicians, actors, etc.) so there was an even playing field of audience. I also would double check that all of the accounts are pretty active in how much they post as that could possibly effect that calculations.